In [1]:
pip install xml_to_dict

In [2]:
pip install xlsxwriter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.0/153.0 kB 4.0 MB/s eta 0:00:00


In [3]:
import requests
import pandas as pd
import numpy as np
from xml_to_dict import XMLtoDict
xd = XMLtoDict()
from datetime import datetime, timedelta
#채권목록 받기
def get_all_bond(cust_no, api_key):
    url = f'http://seibro.or.kr/OpenPlatform/callOpenAPI.jsp?key={api_key}&apiId=getBondIssuInfo&params=ISSUCO_CUSTNO:{cust_no}'
    raw = requests.get(url, verify = False)
    data_dict = xd.parse(raw.content.decode('utf-8'))
    result = data_dict['SeibroAPI']['vector']['@result']
    if result == '0':
        df_all = pd.DataFrame()
    elif result == '1':
        data_list = data_dict['SeibroAPI']['vector']['data']
        df_all = pd.DataFrame(data_list['result'])
    else:
        data_list = data_dict['SeibroAPI']['vector']['data']
        df_all = pd.DataFrame([{k: v['@value'] for k, v in item['result'].items()} for item in data_list])
    return df_all

#채권종목상세내역 받기
def get_specific_bond(ISIN, api_key):
    url = f'http://seibro.or.kr/OpenPlatform/callOpenAPI.jsp?key={api_key}&apiId=getBondStatInfo&params=ISIN:{ISIN}'
    raw = requests.get(url, verify = False)
    data_dict = xd.parse(raw.content.decode('utf-8'))
    result = data_dict['SeibroAPI']['vector']['@result']
    if result == '0':
        df_specific = pd.DataFrame()
    else:
        data_list = data_dict['SeibroAPI']['vector']['data']['result']
        df_specific = pd.DataFrame({k: v['@value'] for k, v in data_list.items()}, index=[0])
        df_specific['ISIN'] = ISIN
    return df_specific

#조기상환내역 API로 받기
def get_redeem(ISIN, api_key):

    url = f'http://seibro.or.kr/OpenPlatform/callOpenAPI.jsp?key={api_key}&apiId=getBondOptionXrcInfo&params=ISIN:{ISIN}'
    raw = requests.get(url, verify = False)
    data_dict = xd.parse(raw.content.decode('utf-8'))
    result = data_dict['SeibroAPI']['vector']['@result']
    if result == '0':
        df_refund_data = pd.DataFrame()
    elif result == '1':
        data_list = data_dict['SeibroAPI']['vector']['data']
        df_refund_data = pd.DataFrame(data_list['result'])
    else:
        data_list = data_dict['SeibroAPI']['vector']['data']
        df_refund_data = pd.DataFrame([{k: v['@value'] for k, v in item['result'].items()} for item in data_list])
    return df_refund_data

#주식관련사채 행사내역(ISIN)
def get_bondstock(ISIN, api_key):
    url = f'http://seibro.or.kr/OpenPlatform/callOpenAPI.jsp?key={api_key}&apiId=getXrcStkOptionXrcInfo&params=BOND_ISIN:{ISIN}'
    raw = requests.get(url, verify = False)
    data_dict = xd.parse(raw.content.decode('utf-8'))
    result = data_dict['SeibroAPI']['vector']['@result']
    if result == '0':
        df_bondstock = pd.DataFrame()
    elif result == '1':
        data_list = data_dict['SeibroAPI']['vector']['data']
        df_bondstock = pd.DataFrame(data_list['result'])
    else:
        data_list = data_dict['SeibroAPI']['vector']['data']
        df_bondstock = pd.DataFrame([{k: v['@value'] for k, v in item['result'].items()} for item in data_list])
    if len(df_bondstock) > 0:
        df_bondstock['채권_종목코드'] = df_bondstock['BOND_ISIN']
        df_bondstock['한글종목명'] = df_bondstock['BOND_KOR_SECN_NM']
        df_bondstock['채권종류코드'] = df_bondstock['BOND_KIND_NM']
        df_bondstock['행사주식_종목코드'] = df_bondstock['XRC_STK_ISIN']
        df_bondstock['행사주식명'] = df_bondstock['STK_KOR_SECN_NM']
        df_bondstock['권리행사일'] = df_bondstock['RGT_STD_DT']
        df_bondstock['행사시작일'] = df_bondstock['XRC_POSS_BEGIN_DT']
        df_bondstock['행사종료일'] = df_bondstock['XRC_POSS_EXPRY_DT']
        df_bondstock['행사금액'] = df_bondstock['XRC_AMT']
        df_bondstock['행사주수'] = df_bondstock['XRC_QTY']
        df_bondstock['행사가격'] = df_bondstock['XRC_PRICE']
        df_bondstock['주식상장일'] = df_bondstock['LIST_DT']
        df_bondstock = df_bondstock[['채권_종목코드', '한글종목명', '채권종류코드', '행사주식_종목코드', '행사주식명', '권리행사일', '행사시작일','행사종료일', '행사금액', '행사주수', '행사가격', '주식상장일']]
    return df_bondstock

#발행채권별 종목정보와 상환내역 받기
def get_details_bond(df_all,api_key):
    #비어있는 데이터프레임 만들기
    df_data = pd.DataFrame()
    df_refund = pd.DataFrame()
    df_bondstock = pd.DataFrame()
    for i in range(len(df_all)):
        ISIN_loop = df_all.ISIN[i]
        #종목별 상세내역 API로 받기
        df_loop = get_specific_bond(ISIN_loop, api_key)
        df_data = pd.concat([df_data, df_loop])
        df_data.reset_index(drop=True ,inplace = True)
        #조기상환내역 API로 받기
        df_refund_loop = get_redeem(ISIN_loop, api_key)
        df_refund = pd.concat([df_refund, df_refund_loop])
        df_refund.reset_index(drop=True ,inplace = True)
        #주식관련사채 행사내역 API로 받기
        df_bondstock_loop = get_bondstock(ISIN_loop, api_key)
        df_bondstock = pd.concat([df_bondstock, df_bondstock_loop])
        df_bondstock.reset_index(drop=True ,inplace = True)


    if len(df_data) >0:
        df_data['모집방법코드'] = np.where(df_data['RECU_WHCD'] == '11', '공모',
                                     np.where(df_data['RECU_WHCD'] == '12', '사모',
                                              np.where(df_data['RECU_WHCD'] == '13', '일반(국채,지방채,특수채)',
                                                       np.where(df_data['RECU_WHCD'] == '14', 'CBO기초사모',
                                                                np.where(df_data['RECU_WHCD'] == '21', '매출',df_data['RECU_WHCD'])))))

        df_data['발행방법코드'] = np.where(df_data['ISSU_WHCD'] == '1', '실물발행',
                                     np.where(df_data['ISSU_WHCD'] == '2', '전액등록',
                                              np.where(df_data['ISSU_WHCD'] == '3', '부분등록',
                                                       np.where(df_data['ISSU_WHCD'] == '4', '청약증거금영수증발행',
                                                                np.where(df_data['ISSU_WHCD'] == '5', '전액불소지',
                                                                         np.where(df_data['ISSU_WHCD'] == '6', '부분불소지', df_data['ISSU_WHCD']))))))

        df_data['특이채권종류구분코드'] = np.where(df_data['PARTICUL_BOND_KIND_TPCD'] == '1', '전환',
                                     np.where(df_data['PARTICUL_BOND_KIND_TPCD'] == '2', '교환',
                                              np.where(df_data['PARTICUL_BOND_KIND_TPCD'] == '3', '신주인수권',
                                                       np.where(df_data['PARTICUL_BOND_KIND_TPCD'] == '4', '분리형신주인수권',
                                                                np.where(df_data['PARTICUL_BOND_KIND_TPCD'] == '6', '이익참가',
                                                                         np.where(df_data['PARTICUL_BOND_KIND_TPCD'] == '9', '해당없음',df_data['PARTICUL_BOND_KIND_TPCD']))))))

        df_data['옵션구분코드'] = np.where(df_data['OPTION_TPCD'] == '9401', 'CALL',
                                     np.where(df_data['OPTION_TPCD'] == '9402', 'PUT',
                                              np.where(df_data['OPTION_TPCD'] == '9403', 'CALL+PUT',
                                                       np.where(df_data['OPTION_TPCD'] == '9404', 'NOTE',
                                                                np.where(df_data['OPTION_TPCD'] == '0000', '옵션해당없음',df_data['OPTION_TPCD'])))))

        df_data['강제조기상환여부'] = df_data['FORC_ERLY_RED_YN']

        df_data['금리변동구분코드'] = np.where(df_data['MR_CHG_TPCD'] == '1', '고정',
                                     np.where(df_data['MR_CHG_TPCD'] == '2', '변동',
                                              np.where(df_data['MR_CHG_TPCD'] == '3', '고정+변동',df_data['MR_CHG_TPCD'])))

        df_data['보증구분코드'] = np.where(df_data['GRTY_TPCD'] == '1', '보증',
                                     np.where(df_data['GRTY_TPCD'] == '2', '무보증',
                                              np.where(df_data['GRTY_TPCD'] == '3', '담보부+PUT',
                                                       np.where(df_data['GRTY_TPCD'] == '4', '일반',df_data['GRTY_TPCD']))))
        df_data['등록기관구분코드'] = np.where(df_data['REGI_ORG_TPCD'] == '1', '예탁원',df_data['REGI_ORG_TPCD'])

        df_data['기명구분코드'] = np.where(df_data['SIGNA_TPCD'] == '1', '기명',
                                     np.where(df_data['SIGNA_TPCD'] == '2', '무기명',
                                              np.where(df_data['SIGNA_TPCD'] == '3', '기명+무기명',df_data['SIGNA_TPCD'])))

        df_data['순위구분코드'] = np.where(df_data['RANK_TPCD'] == '1', '선순위',
                                     np.where(df_data['RANK_TPCD'] == '2', '후순위',
                                              np.where(df_data['RANK_TPCD'] == '3', '중순위',
                                                       np.where(df_data['RANK_TPCD'] == '9', '해당없음',df_data['RANK_TPCD']))))

        df_data['이자지급방법구분코드'] = np.where(df_data['INT_PAY_WAY_TPCD'] == '1', '이표',
                                         np.where(df_data['INT_PAY_WAY_TPCD'] == '2', '할인',
                                              np.where(df_data['INT_PAY_WAY_TPCD'] == '3', '복리',
                                                       np.where(df_data['INT_PAY_WAY_TPCD'] == '4', '단리',df_data['INT_PAY_WAY_TPCD']))))

        df_data['단리복리구분코드'] = np.where(df_data['SINT_CINT_TPCD'] == '1', '단리',
                                         np.where(df_data['SINT_CINT_TPCD'] == '2', '복리',
                                              np.where(df_data['SINT_CINT_TPCD'] == '3', '단리+복리',df_data['SINT_CINT_TPCD'])))

        df_data['이자율변동구분코드'] = np.where(df_data['IRATE_CHG_TPCD'] == '1', '이율동일',
                                         np.where(df_data['IRATE_CHG_TPCD'] == '2', '이율상이',
                                              np.where(df_data['IRATE_CHG_TPCD'] == '3', '비정형',df_data['IRATE_CHG_TPCD'])))

        df_data['만기보장수익율'] = df_data['XPIR_GUAR_PRATE']

        df_data['만기보장수익율구분코드'] = np.where(df_data['XPIR_GUAR_PRATE_TPCD'] == '01', '연복리',
                                              np.where(df_data['XPIR_GUAR_PRATE_TPCD'] == '02', '3개월복리',
                                                       np.where(df_data['XPIR_GUAR_PRATE_TPCD'] == '03', '6개월복리',
                                                                np.where(df_data['XPIR_GUAR_PRATE_TPCD'] == '04', '연단리',
                                                                         np.where(df_data['XPIR_GUAR_PRATE_TPCD'] == '05', '1개월복리',
                                                                                  np.where(df_data['XPIR_GUAR_PRATE_TPCD'] == '99', '해당없음',df_data['XPIR_GUAR_PRATE_TPCD']))))))

        df_data['원금상환방법코드'] = np.where(df_data['PRCP_RED_WHCD'] == '11', '만기상환',
                                              np.where(df_data['PRCP_RED_WHCD'] == '21', '중도상환',
                                                       np.where(df_data['PRCP_RED_WHCD'] == '31', '조기상환',
                                                                np.where(df_data['PRCP_RED_WHCD'] == '41', '이익분배',
                                                                         np.where(df_data['PRCP_RED_WHCD'] == '51', '자동상환',
                                                                                  np.where(df_data['PRCP_RED_WHCD'] == '14', '수시상환',
                                                                                           np.where(df_data['PRCP_RED_WHCD'] == '12', '균등분할상환',
                                                                                                    np.where(df_data['PRCP_RED_WHCD'] == '13', '불균등분할상환',df_data['PRCP_RED_WHCD']))))))))
        df_data['상장일'] = df_data['APLI_DT']
        df_data['상장폐지일자'] = df_data['DLIST_DT']

        df_data['KIS평가등급코드'] = np.where(df_data['KIS_VALAT_GRD_CD'] == '110', 'AAA',
                                              np.where(df_data['KIS_VALAT_GRD_CD'] == '111', 'AAA+',
                                                       np.where(df_data['KIS_VALAT_GRD_CD'] == '112', 'AAA0(A1)',
                                                                np.where(df_data['KIS_VALAT_GRD_CD'] == '113', 'AAA-',
                                                                         np.where(df_data['KIS_VALAT_GRD_CD'] == '120', 'AA',
                                                                                  np.where(df_data['KIS_VALAT_GRD_CD'] == '121', 'AA+(A2+)',
                                                                                           np.where(df_data['KIS_VALAT_GRD_CD'] == '122', 'AA0(A2)',
                                                                                                    np.where(df_data['KIS_VALAT_GRD_CD'] == '123', 'AA-(A2-)',
                                                                                                        np.where(df_data['KIS_VALAT_GRD_CD'] == '130', 'A',
                                                                                                                 np.where(df_data['KIS_VALAT_GRD_CD'] == '131', 'A+(A3+)',
                                                                                                                          np.where(df_data['KIS_VALAT_GRD_CD'] == '132', 'A0(A3)',
                                                                                                                                   np.where(df_data['KIS_VALAT_GRD_CD'] == '133', 'A-(A3-)',
                                                                                                                                            np.where(df_data['KIS_VALAT_GRD_CD'] == '123', 'AA-(A2-)',
                                                                                                                                                np.where(df_data['KIS_VALAT_GRD_CD'] == '210', 'BBB',
                                                                                                                                                         np.where(df_data['KIS_VALAT_GRD_CD'] == '211', 'BBB+',
                                                                                                                                                                  np.where(df_data['KIS_VALAT_GRD_CD'] == '212', 'BBB0',
                                                                                                                                                                           np.where(df_data['KIS_VALAT_GRD_CD'] == '213', 'BBB-',
                                                                                                                                                                                np.where(df_data['KIS_VALAT_GRD_CD'] == '220', 'BB',
                                                                                                                                                                                         np.where(df_data['KIS_VALAT_GRD_CD'] == '221', 'BB+',
                                                                                                                                                                                                  np.where(df_data['KIS_VALAT_GRD_CD'] == '222', 'BB0',
                                                                                                                                                                                                           np.where(df_data['KIS_VALAT_GRD_CD'] == '223', 'BB-',
                                                                                                                                                                                                                    np.where(df_data['KIS_VALAT_GRD_CD'] == '230', 'B',
                                                                                                                                                                                                                             np.where(df_data['KIS_VALAT_GRD_CD'] == '231', 'B+',
                                                                                                                                                                                                                                      np.where(df_data['KIS_VALAT_GRD_CD'] == '232', 'B0(B)',
                                                                                                                                                                                                                                           np.where(df_data['KIS_VALAT_GRD_CD'] == '233', 'B-',
                                                                                                                                                                                                                                                    np.where(df_data['KIS_VALAT_GRD_CD'] == '310', 'CCC',
                                                                                                                                                                                                                                                            np.where(df_data['KIS_VALAT_GRD_CD'] == '311', 'CCC+',
                                                                                                                                                                                                                                                                     np.where(df_data['KIS_VALAT_GRD_CD'] == '312', 'CCC0',
                                                                                                                                                                                                                                                                              np.where(df_data['KIS_VALAT_GRD_CD'] == '313', 'CCC-',
                                                                                                                                                                                                                                                                                       np.where(df_data['KIS_VALAT_GRD_CD'] == '320', 'CC',
                                                                                                                                                                                                                                                                                                np.where(df_data['KIS_VALAT_GRD_CD'] == '321', 'CC+',
                                                                                                                                                                                                                                                                                                         np.where(df_data['KIS_VALAT_GRD_CD'] == '322', 'CC0',
                                                                                                                                                                                                                                                                                                                  np.where(df_data['KIS_VALAT_GRD_CD'] == '323', 'CC-',
                                                                                                                                                                                                                                                                                                                           np.where(df_data['KIS_VALAT_GRD_CD'] == '330', 'C',
                                                                                                                                                                                                                                                                                                                                    np.where(df_data['KIS_VALAT_GRD_CD'] == '331', 'C+',
                                                                                                                                                                                                                                                                                                                                             np.where(df_data['KIS_VALAT_GRD_CD'] == '332', 'C0',
                                                                                                                                                                                                                                                                                                                                                      np.where(df_data['KIS_VALAT_GRD_CD'] == '333', 'C-',
                                                                                                                                                                                                                                                                                                                                                               np.where(df_data['KIS_VALAT_GRD_CD'] == '440', 'D',
                                                                                                                                                                                                                                                                                                                                                                        np.where(df_data['KIS_VALAT_GRD_CD'] == '900', '유보',
                                                                                                                                                                                                                                                                                                                                                                                 np.where(df_data['KIS_VALAT_GRD_CD'] == '999', '취소',df_data['KIS_VALAT_GRD_CD']))))))))))))))))))))))))))))))))))))))))

        df_data['NICE평가등급코드'] = np.where(df_data['NICE_VALAT_GRD_CD'] == '110', 'AAA',
                                              np.where(df_data['NICE_VALAT_GRD_CD'] == '111', 'AAA+',
                                                       np.where(df_data['NICE_VALAT_GRD_CD'] == '112', 'AAA0(A1)',
                                                                np.where(df_data['NICE_VALAT_GRD_CD'] == '113', 'AAA-',
                                                                         np.where(df_data['NICE_VALAT_GRD_CD'] == '120', 'AA',
                                                                                  np.where(df_data['NICE_VALAT_GRD_CD'] == '121', 'AA+(A2+)',
                                                                                           np.where(df_data['NICE_VALAT_GRD_CD'] == '122', 'AA0(A2)',
                                                                                                    np.where(df_data['NICE_VALAT_GRD_CD'] == '123', 'AA-(A2-)',
                                                                                                        np.where(df_data['NICE_VALAT_GRD_CD'] == '130', 'A',
                                                                                                                 np.where(df_data['NICE_VALAT_GRD_CD'] == '131', 'A+(A3+)',
                                                                                                                          np.where(df_data['NICE_VALAT_GRD_CD'] == '132', 'A0(A3)',
                                                                                                                                   np.where(df_data['NICE_VALAT_GRD_CD'] == '133', 'A-(A3-)',
                                                                                                                                            np.where(df_data['NICE_VALAT_GRD_CD'] == '123', 'AA-(A2-)',
                                                                                                                                                np.where(df_data['NICE_VALAT_GRD_CD'] == '210', 'BBB',
                                                                                                                                                         np.where(df_data['NICE_VALAT_GRD_CD'] == '211', 'BBB+',
                                                                                                                                                                  np.where(df_data['NICE_VALAT_GRD_CD'] == '212', 'BBB0',
                                                                                                                                                                           np.where(df_data['NICE_VALAT_GRD_CD'] == '213', 'BBB-',
                                                                                                                                                                                np.where(df_data['NICE_VALAT_GRD_CD'] == '220', 'BB',
                                                                                                                                                                                         np.where(df_data['NICE_VALAT_GRD_CD'] == '221', 'BB+',
                                                                                                                                                                                                  np.where(df_data['NICE_VALAT_GRD_CD'] == '222', 'BB0',
                                                                                                                                                                                                           np.where(df_data['NICE_VALAT_GRD_CD'] == '223', 'BB-',
                                                                                                                                                                                                                    np.where(df_data['NICE_VALAT_GRD_CD'] == '230', 'B',
                                                                                                                                                                                                                             np.where(df_data['NICE_VALAT_GRD_CD'] == '231', 'B+',
                                                                                                                                                                                                                                      np.where(df_data['NICE_VALAT_GRD_CD'] == '232', 'B0(B)',
                                                                                                                                                                                                                                           np.where(df_data['NICE_VALAT_GRD_CD'] == '233', 'B-',
                                                                                                                                                                                                                                                    np.where(df_data['NICE_VALAT_GRD_CD'] == '310', 'CCC',
                                                                                                                                                                                                                                                            np.where(df_data['NICE_VALAT_GRD_CD'] == '311', 'CCC+',
                                                                                                                                                                                                                                                                     np.where(df_data['NICE_VALAT_GRD_CD'] == '312', 'CCC0',
                                                                                                                                                                                                                                                                              np.where(df_data['NICE_VALAT_GRD_CD'] == '313', 'CCC-',
                                                                                                                                                                                                                                                                                       np.where(df_data['NICE_VALAT_GRD_CD'] == '320', 'CC',
                                                                                                                                                                                                                                                                                                np.where(df_data['NICE_VALAT_GRD_CD'] == '321', 'CC+',
                                                                                                                                                                                                                                                                                                         np.where(df_data['NICE_VALAT_GRD_CD'] == '322', 'CC0',
                                                                                                                                                                                                                                                                                                                  np.where(df_data['NICE_VALAT_GRD_CD'] == '323', 'CC-',
                                                                                                                                                                                                                                                                                                                           np.where(df_data['NICE_VALAT_GRD_CD'] == '330', 'C',
                                                                                                                                                                                                                                                                                                                                    np.where(df_data['NICE_VALAT_GRD_CD'] == '331', 'C+',
                                                                                                                                                                                                                                                                                                                                             np.where(df_data['NICE_VALAT_GRD_CD'] == '332', 'C0',
                                                                                                                                                                                                                                                                                                                                                      np.where(df_data['NICE_VALAT_GRD_CD'] == '333', 'C-',
                                                                                                                                                                                                                                                                                                                                                               np.where(df_data['NICE_VALAT_GRD_CD'] == '440', 'D',
                                                                                                                                                                                                                                                                                                                                                                        np.where(df_data['NICE_VALAT_GRD_CD'] == '900', '유보',
                                                                                                                                                                                                                                                                                                                                                                                 np.where(df_data['NICE_VALAT_GRD_CD'] == '999', '취소',df_data['NICE_VALAT_GRD_CD']))))))))))))))))))))))))))))))))))))))))

        df_data['서울신용평가등급코드'] = np.where(df_data['SCI_VALAT_GRD_CD'] == '110', 'AAA',
                                              np.where(df_data['SCI_VALAT_GRD_CD'] == '111', 'AAA+',
                                                       np.where(df_data['SCI_VALAT_GRD_CD'] == '112', 'AAA0(A1)',
                                                                np.where(df_data['SCI_VALAT_GRD_CD'] == '113', 'AAA-',
                                                                         np.where(df_data['SCI_VALAT_GRD_CD'] == '120', 'AA',
                                                                                  np.where(df_data['SCI_VALAT_GRD_CD'] == '121', 'AA+(A2+)',
                                                                                           np.where(df_data['SCI_VALAT_GRD_CD'] == '122', 'AA0(A2)',
                                                                                                    np.where(df_data['SCI_VALAT_GRD_CD'] == '123', 'AA-(A2-)',
                                                                                                        np.where(df_data['SCI_VALAT_GRD_CD'] == '130', 'A',
                                                                                                                 np.where(df_data['SCI_VALAT_GRD_CD'] == '131', 'A+(A3+)',
                                                                                                                          np.where(df_data['SCI_VALAT_GRD_CD'] == '132', 'A0(A3)',
                                                                                                                                   np.where(df_data['SCI_VALAT_GRD_CD'] == '133', 'A-(A3-)',
                                                                                                                                            np.where(df_data['SCI_VALAT_GRD_CD'] == '123', 'AA-(A2-)',
                                                                                                                                                np.where(df_data['SCI_VALAT_GRD_CD'] == '210', 'BBB',
                                                                                                                                                         np.where(df_data['SCI_VALAT_GRD_CD'] == '211', 'BBB+',
                                                                                                                                                                  np.where(df_data['SCI_VALAT_GRD_CD'] == '212', 'BBB0',
                                                                                                                                                                           np.where(df_data['SCI_VALAT_GRD_CD'] == '213', 'BBB-',
                                                                                                                                                                                np.where(df_data['SCI_VALAT_GRD_CD'] == '220', 'BB',
                                                                                                                                                                                         np.where(df_data['SCI_VALAT_GRD_CD'] == '221', 'BB+',
                                                                                                                                                                                                  np.where(df_data['SCI_VALAT_GRD_CD'] == '222', 'BB0',
                                                                                                                                                                                                           np.where(df_data['SCI_VALAT_GRD_CD'] == '223', 'BB-',
                                                                                                                                                                                                                    np.where(df_data['SCI_VALAT_GRD_CD'] == '230', 'B',
                                                                                                                                                                                                                             np.where(df_data['SCI_VALAT_GRD_CD'] == '231', 'B+',
                                                                                                                                                                                                                                      np.where(df_data['SCI_VALAT_GRD_CD'] == '232', 'B0(B)',
                                                                                                                                                                                                                                           np.where(df_data['SCI_VALAT_GRD_CD'] == '233', 'B-',
                                                                                                                                                                                                                                                    np.where(df_data['SCI_VALAT_GRD_CD'] == '310', 'CCC',
                                                                                                                                                                                                                                                            np.where(df_data['SCI_VALAT_GRD_CD'] == '311', 'CCC+',
                                                                                                                                                                                                                                                                     np.where(df_data['SCI_VALAT_GRD_CD'] == '312', 'CCC0',
                                                                                                                                                                                                                                                                              np.where(df_data['SCI_VALAT_GRD_CD'] == '313', 'CCC-',
                                                                                                                                                                                                                                                                                       np.where(df_data['SCI_VALAT_GRD_CD'] == '320', 'CC',
                                                                                                                                                                                                                                                                                                np.where(df_data['SCI_VALAT_GRD_CD'] == '321', 'CC+',
                                                                                                                                                                                                                                                                                                         np.where(df_data['SCI_VALAT_GRD_CD'] == '322', 'CC0',
                                                                                                                                                                                                                                                                                                                  np.where(df_data['SCI_VALAT_GRD_CD'] == '323', 'CC-',
                                                                                                                                                                                                                                                                                                                           np.where(df_data['SCI_VALAT_GRD_CD'] == '330', 'C',
                                                                                                                                                                                                                                                                                                                                    np.where(df_data['SCI_VALAT_GRD_CD'] == '331', 'C+',
                                                                                                                                                                                                                                                                                                                                             np.where(df_data['SCI_VALAT_GRD_CD'] == '332', 'C0',
                                                                                                                                                                                                                                                                                                                                                      np.where(df_data['SCI_VALAT_GRD_CD'] == '333', 'C-',
                                                                                                                                                                                                                                                                                                                                                               np.where(df_data['SCI_VALAT_GRD_CD'] == '440', 'D',
                                                                                                                                                                                                                                                                                                                                                                        np.where(df_data['SCI_VALAT_GRD_CD'] == '900', '유보',
                                                                                                                                                                                                                                                                                                                                                                                 np.where(df_data['SCI_VALAT_GRD_CD'] == '999', '취소',df_data['SCI_VALAT_GRD_CD']))))))))))))))))))))))))))))))))))))))))

        df_data['한국기업평가등급코드'] = np.where(df_data['KR_VALAT_GRD_CD'] == '110', 'AAA',
                                              np.where(df_data['KR_VALAT_GRD_CD'] == '111', 'AAA+',
                                                       np.where(df_data['KR_VALAT_GRD_CD'] == '112', 'AAA0(A1)',
                                                                np.where(df_data['KR_VALAT_GRD_CD'] == '113', 'AAA-',
                                                                         np.where(df_data['KR_VALAT_GRD_CD'] == '120', 'AA',
                                                                                  np.where(df_data['KR_VALAT_GRD_CD'] == '121', 'AA+(A2+)',
                                                                                           np.where(df_data['KR_VALAT_GRD_CD'] == '122', 'AA0(A2)',
                                                                                                    np.where(df_data['KR_VALAT_GRD_CD'] == '123', 'AA-(A2-)',
                                                                                                        np.where(df_data['KR_VALAT_GRD_CD'] == '130', 'A',
                                                                                                                 np.where(df_data['KR_VALAT_GRD_CD'] == '131', 'A+(A3+)',
                                                                                                                          np.where(df_data['KR_VALAT_GRD_CD'] == '132', 'A0(A3)',
                                                                                                                                   np.where(df_data['KR_VALAT_GRD_CD'] == '133', 'A-(A3-)',
                                                                                                                                            np.where(df_data['KR_VALAT_GRD_CD'] == '123', 'AA-(A2-)',
                                                                                                                                                np.where(df_data['KR_VALAT_GRD_CD'] == '210', 'BBB',
                                                                                                                                                         np.where(df_data['KR_VALAT_GRD_CD'] == '211', 'BBB+',
                                                                                                                                                                  np.where(df_data['KR_VALAT_GRD_CD'] == '212', 'BBB0',
                                                                                                                                                                           np.where(df_data['KR_VALAT_GRD_CD'] == '213', 'BBB-',
                                                                                                                                                                                np.where(df_data['KR_VALAT_GRD_CD'] == '220', 'BB',
                                                                                                                                                                                         np.where(df_data['KR_VALAT_GRD_CD'] == '221', 'BB+',
                                                                                                                                                                                                  np.where(df_data['KR_VALAT_GRD_CD'] == '222', 'BB0',
                                                                                                                                                                                                           np.where(df_data['KR_VALAT_GRD_CD'] == '223', 'BB-',
                                                                                                                                                                                                                    np.where(df_data['KR_VALAT_GRD_CD'] == '230', 'B',
                                                                                                                                                                                                                             np.where(df_data['KR_VALAT_GRD_CD'] == '231', 'B+',
                                                                                                                                                                                                                                      np.where(df_data['KR_VALAT_GRD_CD'] == '232', 'B0(B)',
                                                                                                                                                                                                                                           np.where(df_data['KR_VALAT_GRD_CD'] == '233', 'B-',
                                                                                                                                                                                                                                                    np.where(df_data['KR_VALAT_GRD_CD'] == '310', 'CCC',
                                                                                                                                                                                                                                                            np.where(df_data['KR_VALAT_GRD_CD'] == '311', 'CCC+',
                                                                                                                                                                                                                                                                     np.where(df_data['KR_VALAT_GRD_CD'] == '312', 'CCC0',
                                                                                                                                                                                                                                                                              np.where(df_data['KR_VALAT_GRD_CD'] == '313', 'CCC-',
                                                                                                                                                                                                                                                                                       np.where(df_data['KR_VALAT_GRD_CD'] == '320', 'CC',
                                                                                                                                                                                                                                                                                                np.where(df_data['KR_VALAT_GRD_CD'] == '321', 'CC+',
                                                                                                                                                                                                                                                                                                         np.where(df_data['KR_VALAT_GRD_CD'] == '322', 'CC0',
                                                                                                                                                                                                                                                                                                                  np.where(df_data['KR_VALAT_GRD_CD'] == '323', 'CC-',
                                                                                                                                                                                                                                                                                                                           np.where(df_data['KR_VALAT_GRD_CD'] == '330', 'C',
                                                                                                                                                                                                                                                                                                                                    np.where(df_data['KR_VALAT_GRD_CD'] == '331', 'C+',
                                                                                                                                                                                                                                                                                                                                             np.where(df_data['KR_VALAT_GRD_CD'] == '332', 'C0',
                                                                                                                                                                                                                                                                                                                                                      np.where(df_data['KR_VALAT_GRD_CD'] == '333', 'C-',
                                                                                                                                                                                                                                                                                                                                                               np.where(df_data['KR_VALAT_GRD_CD'] == '440', 'D',
                                                                                                                                                                                                                                                                                                                                                                        np.where(df_data['KR_VALAT_GRD_CD'] == '900', '유보',
                                                                                                                                                                                                                                                                                                                                                                                 np.where(df_data['KR_VALAT_GRD_CD'] == '999', '취소',df_data['KR_VALAT_GRD_CD']))))))))))))))))))))))))))))))))))))))))

        df_data['발행일자'] = df_data['ISSU_DT']
        df_data['만기일자'] = df_data['XPIR_DT']
        df_data['발행통화코드'] = df_data['ISSU_CUR_CD']
        df_data['발행금액'] = df_data['FIRST_ISSU_AMT']
        df_data['발행잔액'] = df_data['ISSU_REMA']
        df_data['납입금액'] = df_data['PAYIN_AMT']
        df_data['표면이자율'] = df_data['COUPON_RATE']
        df_data['만기상환율'] = df_data['XPIRED_RATE']


        #필요한 칼럼만 남기기
        df_data = df_data[['ISSUCO_CUSTNO','KOR_SECN_NM','SECN_KACD','발행일자','만기일자','발행통화코드','발행금액','발행잔액','납입금액','표면이자율','만기상환율','모집방법코드','발행방법코드','특이채권종류구분코드','옵션구분코드','강제조기상환여부','금리변동구분코드','등록기관구분코드','보증구분코드','기명구분코드','순위구분코드','이자지급방법구분코드','단리복리구분코드','이자율변동구분코드','만기보장수익율','만기보장수익율구분코드','원금상환방법코드','상장일','상장폐지일자','KIS평가등급코드','NICE평가등급코드','서울신용평가등급코드','한국기업평가등급코드']]

    if len(df_refund) > 0:

        df_refund['옵션구분코드'] = np.where(df_refund['OPTION_TPCD'] == '9401', 'CALL',
                                     np.where(df_refund['OPTION_TPCD'] == '9402', 'PUT',
                                              np.where(df_refund['OPTION_TPCD'] == '9403', 'CALL+PUT',df_refund['OPTION_TPCD'])))

        df_refund['조기상환일'] = df_refund['ERLY_RED_DT']
        df_refund['적용이자율'] = df_refund['APLI_IRATE']
        df_refund['조기상환금액'] = df_refund['ERLY_REDAMT_VAL']
        df_refund['이자지급금액'] = df_refund['INT_PAY_AMT']
        df_refund['최근발행잔액'] = df_refund['ISSU_REMA']
        df_refund['행사비율'] = df_refund['XRC_RATIO']

        #필요한 칼럼만 남기기
        df_refund = df_refund[['ISIN','KOR_SECN_NM','옵션구분코드','조기상환일','조기상환금액','최근발행잔액']]
    return [df_data, df_refund, df_bondstock]


stock_code = input('종목코드를 입력하세요 : ')


import requests
import pandas as pd
import numpy as np
from xml_to_dict import XMLtoDict
xd = XMLtoDict()
from datetime import datetime, timedelta


api_key = '752127f8d9bed7220dbc9dfb1b67610181e7689aee0e03274589f472a8324cd7'


def get_all_bond(cust_no, api_key):
    url = f'http://seibro.or.kr/OpenPlatform/callOpenAPI.jsp?key={api_key}&apiId=getBondIssuInfo&params=ISSUCO_CUSTNO:{cust_no}'
    raw = requests.get(url, verify = False)
    data_dict = xd.parse(raw.content.decode('utf-8'))
    result = data_dict['SeibroAPI']['vector']['@result']
    if result == '0':
        df_all = pd.DataFrame()
    elif result == '1':
        data_list = data_dict['SeibroAPI']['vector']['data']
        df_all = pd.DataFrame(data_list['result'])
    else:
        data_list = data_dict['SeibroAPI']['vector']['data']
        df_all = pd.DataFrame([{k: v['@value'] for k, v in item['result'].items()} for item in data_list])
    return df_all

#종목코드로 예탁원 코드 가져올 수 있는 데이터프레임 만들기
#11유가, 12코스닥, 14코넥스
mkt_list = ['11','12']
mkt = mkt_list[1]
url = f'http://seibro.or.kr/OpenPlatform/callOpenAPI.jsp?key={api_key}&apiId=getShotnByMart&params=MART_TPCD:{mkt}'
raw = requests.get(url, verify = False)
data_dict = xd.parse(raw.content.decode('utf-8'))
data_list = data_dict['SeibroAPI']['vector']['data']
records = []
for item in data_list:
    record = {
        'SHOTN_ISIN': item['result']['SHOTN_ISIN']['@value'],
        'KOR_SECN_NM': item['result']['KOR_SECN_NM']['@value'],
        'ISSUCO_CUSTNO': item['result']['ISSUCO_CUSTNO']['@value']
    }
    records.append(record)
df_stock_cust = pd.DataFrame(records)

###
###
###
###
###
###
###


#종목코드 입력하면 예탁원 코드 반환하여 cust_no로 저장
cust_no = df_stock_cust[df_stock_cust.SHOTN_ISIN == stock_code]['ISSUCO_CUSTNO'].item()
stock_name = df_stock_cust[df_stock_cust.SHOTN_ISIN == stock_code]['KOR_SECN_NM'].item()

#주식_상장정보(미사용)
#start_dt = '20230704'
#end_dt = '20230713'
#url = f'http://seibro.or.kr/OpenPlatform/callOpenAPI.jsp?key={api_key}&apiId=getStkListInfo&params=ALT_BEGIN_DT:{start_dt},ALT_EXPRY_DT:{end_dt}'
#raw = requests.get(url, verify = False)
#data_dict = xd.parse(raw.content.decode('utf-8'))
#data_list = data_dict['SeibroAPI']['vector']['data']
#records = []
#for item in data_list:
#    result_data = item['result']
#    record = {
#        'ISSUCO_CUSTNO': result_data['ISSUCO_CUSTNO']['@value'],
#        'ISIN': result_data['ISIN']['@value'],
#        'SHOTN_ISIN': result_data['SHOTN_ISIN']['@value'],
#        'KOR_SECN_NM': result_data['KOR_SECN_NM']['@value'],
#        'BFALT_CIRCL_FORM': result_data['BFALT_CIRCL_FORM']['@value'],
#        'AFALT_CIRCL_FORM': result_data['AFALT_CIRCL_FORM']['@value'],
#        'APLI_DT': result_data['APLI_DT']['@value']
#    }
#    records.append(record)
#
# Create a DataFrame from the list of dictionaries
#df_list_info = pd.DataFrame(records)

#상장사의 전체 채권 내역
df_all_bond = get_all_bond(cust_no, api_key)
df_all_bond.reset_index(drop= True, inplace=True)

#get_details_bond는 다음을 반환
#1. 채권종목 상세내역
#2. 조기상환 상세내역
#3. 주식관련사채 행사내역
list_df = get_details_bond(df_all_bond,api_key)
df1 = list_df[0]
df2 = list_df[1]
df3 = list_df[2]

#열의 데이터 형태 바꿔주기
df1_int_col = ['발행금액', '발행잔액', '납입금액']
df1_date_col = ['발행일자', '만기일자','상장일', '상장폐지일자']
df2_int_col = ['조기상환금액', '최근발행잔액']
df2_date_col = ['조기상환일']
df3_int_col = ['행사금액', '행사주수', '행사가격']
df3_date_col = ['권리행사일', '행사시작일','행사종료일','주식상장일']

if len(df1)>0:
    df1[df1_date_col] = df1[df1_date_col].applymap(lambda x: pd.to_datetime(x, format='%Y%m%d', errors='ignore'))
    df1[df1_int_col] = df1[df1_int_col].astype(np.int64, errors='ignore')
else :
    print(f'{stock_code}_{stock_name}는 SEIBRO에 등록된 채권종목이 없습니다.')
if len(df2)>0:
    df2[df2_date_col] = df2[df2_date_col].applymap(lambda x: pd.to_datetime(x, format='%Y%m%d', errors='ignore'))
    df2[df2_int_col] = df2[df2_int_col].astype(np.int64, errors='ignore')
else :
    print(f'{stock_code}_{stock_name}는 SEIBRO에 등록된 채권조기상환 내역이 없습니다.')
if len(df3)>0:
    df3[df3_date_col] = df3[df3_date_col].applymap(lambda x: pd.to_datetime(x, format='%Y%m%d', errors='ignore'))
    df3[df3_int_col] = df3[df3_int_col].astype(np.int64, errors='ignore')
else :
    print(f'{stock_code}_{stock_name}는 SEIBRO에 등록된 주식관련사채 행사 내역이 없습니다.')

#보호예수 정보 취득하는 함수
#오늘 기준으로 최근 2년간 보호예수 및 반환 내역 불러오는 함수
def get_deposit(cust_no, api_key):
    #검색종료일(end_dt)을 오늘로 설정하면
    #start_dt는 검색종료일-1년 +1일로 설정
    end_dt_tmp = datetime.today()
    end_dt = end_dt_tmp.strftime('%Y%m%d')
    start_dt_tmp = end_dt_tmp.replace(year=end_dt_tmp.year - 1)
    start_dt_tmp = start_dt_tmp + timedelta(days=1)
    start_dt = start_dt_tmp.strftime('%Y%m%d')

    end_dt2_tmp = start_dt_tmp - timedelta(days=1)
    end_dt2 = end_dt2_tmp.strftime('%Y%m%d')

    start_dt2_tmp = end_dt2_tmp.replace(year=end_dt2_tmp.year - 1)
    start_dt2_tmp = start_dt2_tmp + timedelta(days=1)
    start_dt2 = start_dt2_tmp.strftime('%Y%m%d')
    #날짜리스트 생성
    start_date_list = [start_dt, start_dt2]
    end_date_list = [end_dt, end_dt2]

    #'1'은 예탁, '2'는 반환
    biz_list = ['1','2']
    #코드별 내용을 데이터프레임의로 변환
    df_dict = {'code':['00','01','02','03','04','43','05','06','07','08','09','10','11','12','13','14','15','16','17','18','19','20','21','44','22','23','24','25','26','27','28','29','30','31','32','33','34','35','36','37','38','39','40','48','41','42','45','46','49','47','50','51','52','90','99','53','54','55'],
               '보호예수사유':['해당없음','일반','모집(전매제한)','최대주주(상장)','최대주주(코스닥)','최대주주(기술성장기업)','투자회사(상장)','초과유상증자','지정취소','질권담보','등록주선인','외국인양수(변동제한)','벤처금융','기관투자가','합병(상장)','합병(코스닥)','부동산(상장)','부동산(코스닥)','법원M&A','주식교환(코스닥)','벤처금융(주식교환)','벤처금융(합병)','기관투자가(합병)','기관투자가(코넥스)','기관투자가(주식교환)','제3자배정최대주주(코스닥)','선박(상장)','채권기관협의회','주식교환(상장)','영업자산양도&제3자배정(상장)','영업자산양도&제3자배정(코스닥)','제3자배정유상증자(상장)','제3자배정유상증자(코스닥)','최대주주(제3자배정)','SPAC(상장)','SPAC(코스닥)','SPAC합병(상장)','SPAC합병(코스닥)','명목회사(상장)','명목회사(코스닥)','자발적보호예수','우회상장','상장주선인(국내기업)','상장주선인(외국기업)','전매제한','기타(특정금전신탁편입등)','크라우드펀딩','크라우드펀딩(최대주주)','모집(전매제한)-크라우드펀딩전문투자자','주관회사(금투협)','상장주선인(50%미만)','상장주선인(50%이상)','최대주주변경(법인또는','기타의무예수','기타','상장적격성실질심사','기타보호예시필요주주','주식매수선택권']}
    df_code_list = pd.DataFrame(df_dict)

    #비어있는데이터프레임
    df_deposit_f = pd.DataFrame()

    #예탁 및 반환
    for biz in biz_list :
        #최근 2년간
        for start_dt, end_dt in zip(start_date_list, end_date_list):
            url = f'http://seibro.or.kr/OpenPlatform/callOpenAPI.jsp?key={api_key}&apiId=getSafeDpDutyDepoStatus&params=ISSUCO_CUSTNO:{cust_no},BEGIN_DT:{start_dt},EXPRY_DT:{end_dt},BIZ_TPCD:{biz}'
            raw = requests.get(url, verify = False)
            data_dict = xd.parse(raw.content.decode('utf-8'))
            result = data_dict['SeibroAPI']['vector']['@result']
            if result == '0':
                df_deposit = pd.DataFrame()
            elif result == '1':
                data_list = data_dict['SeibroAPI']['vector']['data']
                df_deposit = pd.DataFrame(data_list['result'])
            else:
                data_list = data_dict['SeibroAPI']['vector']['data']
                df_deposit = pd.DataFrame([{k: v['@value'] for k, v in item['result'].items()} for item in data_list])

            if len(df_deposit) > 0:
                df_deposit = pd.merge(df_deposit, df_code_list, how = 'left',
                                      left_on = 'DUTY_SAFEDP_RACD', right_on = 'code')
                df_deposit.drop('DUTY_SAFEDP_RACD', axis = 1, inplace = True)

                df_deposit['종목종류'] = df_deposit['SECN_KACD']
                df_deposit['단축코드'] = df_deposit['SHOTN_ISIN']
                df_deposit['시장구분'] = np.where(df_deposit['MART_TPCD'] == '11', '유가',
                                                 np.where(df_deposit['MART_TPCD'] == '12', '코스닥',
                                                      np.where(df_deposit['MART_TPCD'] == '14', '코넥스',df_deposit['MART_TPCD'])))

                df_deposit['업무구분'] = np.where(df_deposit['OCCR_SEQ'] == '1', '예수',
                                                 np.where(df_deposit['OCCR_SEQ'] == '2', '반환',df_deposit['OCCR_SEQ']))
                df_deposit['예수일'] = df_deposit['SAFEDP_DT']
                df_deposit['예수주식수'] = df_deposit['SAFEDP_QTY']
                df_deposit['반환일'] = df_deposit['RETURN_DT']
                df_deposit['반환주식수'] = df_deposit['RETURN_QTY']
                df_deposit['총발행주식수'] = df_deposit['TOTAL_STK_CNT']

                df_deposit = df_deposit[['ISSUCO_CUSTNO','단축코드','KOR_SECN_NM','종목종류',
                                         '시장구분', '업무구분', '보호예수사유','예수일', '예수주식수', '반환일', '반환주식수', '총발행주식수']]

                int_col = ['예수주식수', '반환주식수', '총발행주식수']
                date_col = ['예수일', '반환일']

                df_deposit[date_col] = df_deposit[date_col].applymap(lambda x: pd.to_datetime(x, format='%Y%m%d', errors='ignore'))
                df_deposit[int_col] = df_deposit[int_col].astype(np.int64, errors='ignore')

                #데이터프레임 결합
                df_deposit_f = pd.concat([df_deposit_f, df_deposit])

    return df_deposit_f

#횟수별
#주식증감내역(stock_code)
def stock_change(stock_code, api_key):
    url = f'http://seibro.or.kr/OpenPlatform/callOpenAPI.jsp?key={api_key}&apiId=getStkIncdecDetails&params=SHOTN_ISIN:{stock_code}'
    raw = requests.get(url, verify = False)
    data_dict = xd.parse(raw.content.decode('utf-8'))
    result = data_dict['SeibroAPI']['vector']['@result']
    if result == '0':
        df_stock_change = pd.DataFrame()
    elif result == '1':
        data_list = data_dict['SeibroAPI']['vector']['data']
        df_stock_change = pd.DataFrame(data_list['result'])
    else:
        data_list = data_dict['SeibroAPI']['vector']['data']
        df_stock_change = pd.DataFrame([{k: v['@value'] for k, v in item['result'].items()} for item in data_list])

    code_dict = {'code' : ['0101' , '0201', '0301' , '0401' ],
                 '종목종류' : ['보통주', '우선주', '후배주', '혼합주']}
    df_code = pd.DataFrame(code_dict)

    if len(df_stock_change) > 0:

        df_stock_change = pd.merge(df_stock_change, df_code, how = 'left',
                              left_on = 'SECN_KACD', right_on = 'code')
        df_stock_change.drop('SECN_KACD', axis = 1, inplace = True)


        df_stock_change['종목발행횟수'] = df_stock_change['SECN_ISSU_NTIMES']
        df_stock_change['발행일자'] = df_stock_change['ISSU_DT']
        df_stock_change['발행가'] = df_stock_change['ISSUPRC']
        df_stock_change['발행수량'] = df_stock_change['ISSU_QTY']
        df_stock_change['종목발행사유코드'] = df_stock_change['SECN_ISSU_RACD']
        df_stock_change['종목발행사유명'] = df_stock_change['SECN_ISSU_NM']
        df_stock_change['상장일자'] = df_stock_change['LIST_DT']
        df_stock_change = df_stock_change[['ISSUCO_CUSTNO','ISIN','종목종류','종목발행횟수',
                                           '발행일자', '발행가', '발행수량', '종목발행사유명', '상장일자']]

        int_col = ['종목발행횟수', '발행가', '발행수량']
        date_col = ['발행일자', '상장일자']

        df_stock_change[date_col] = df_stock_change[date_col].applymap(lambda x: pd.to_datetime(x, format='%Y%m%d', errors='ignore'))
        df_stock_change[int_col] = df_stock_change[int_col].astype(np.int64, errors='ignore')
    return df_stock_change

#횟수별 주식발행내역
df_stock_ch = stock_change(stock_code, api_key)
#의무보유내역
df_deposit = get_deposit(cust_no, api_key)

excel_writer = pd.ExcelWriter(f'{stock_code}_{stock_name}.xlsx', engine='xlsxwriter')
df_stock_ch.to_excel(excel_writer, sheet_name=f'{stock_name}_횟수별발행내역', index=False)
df_deposit.to_excel(excel_writer, sheet_name=f'{stock_name}_의무보유내역', index=False)
df1.to_excel(excel_writer, sheet_name=f'{stock_name}_채권종목정보', index=False)
df2.to_excel(excel_writer, sheet_name=f'{stock_name}_채권조기상환', index=False)
df3.to_excel(excel_writer, sheet_name=f'{stock_name}_주식관련사채행사', index=False)

excel_writer.save()

from google.colab import files

files.download(f'{stock_code}_{stock_name}.xlsx')


#주식증감내역(stock_code)
# def stock_change(ISIN, api_key):
#     url = f'http://seibro.or.kr/OpenPlatform/callOpenAPI.jsp?key={api_key}&apiId=getStkIncdecDetails&params=SHOTN_ISIN:{stock_code}'
#     raw = requests.get(url, verify = False)
#     data_dict = xd.parse(raw.content.decode('utf-8'))
#     result = data_dict['SeibroAPI']['vector']['@result']
#     if result == '0':
#         df_stock_change = pd.DataFrame()
#     elif result == '1':
#         data_list = data_dict['SeibroAPI']['vector']['data']
#         df_stock_change = pd.DataFrame(data_list['result'])
#     else:
#         data_list = data_dict['SeibroAPI']['vector']['data']
#         df_stock_change = pd.DataFrame([{k: v['@value'] for k, v in item['result'].items()} for item in data_list])
#     return df_stock_change




종목코드를 입력하세요 : 096630


/usr/local/lib/python3.10/dist-packages/urllib3/connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'seibro.or.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/urllib3/connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'seibro.or.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/urllib3/connectionpool.py:1056: InsecureRequestWarning: Unverified HTTPS request is being made to host 'seibro.or.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/urllib3/connectionpool.py:1056: Inse

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>